# 63 — Structured Stochastic Processes (StoP)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Stochastic Processes (StoP)
**Blueprint:** `outputs/reports/structured/structured_stop_blueprint.md` — read this first. This is
deliberately NOT a port of DS5's StoP notebook: DS5 modeled a bounded daily override RATE; the theory
paper's own StoP example for Structured is an unbounded daily COUNT ("the number of timed tasks created
per user-day"). Same method-of-moments estimation technique, different paper-justified state variable.
**Output root:** `outputs/model_outputs/structured/stochastic_processes/`

---

## Two daily state variables (both named in the paper's Temporal Role sentence)

1. `N_t^created` — count of `timed`-task `Task Created` events per participant-LOCAL-day
2. `N_t^completed` — count of `task_completed` marking events per participant-LOCAL-day

## Key design decisions (see blueprint for full justification)

- **Local-calendar-day binning** via each event's own `event_timezone`, not UTC — verified necessary:
  24/61 participants have >1 distinct timezone across their events.
- **Zero-filled regular daily grid** from first to last local-active-day per series — a day with 0
  creations/completions is a real zero, not missing data.
- **Eligibility:** `MIN_SPAN_DAYS=14` and `MIN_ACTIVE_DAYS=5`, verified against this dataset's own
  local-day distributions (not DS5's differently-defined `MIN_DAYS=7`).
- **New metric not used in DS5:** Fano factor (Var/Mean of the raw count series) as a count-process-
  appropriate burstiness measure, directly operationalizing the paper's "bursty" phenotype language.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "stochastic_processes"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MIN_SPAN_DAYS = 14
MIN_ACTIVE_DAYS = 5

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
print(f"Behavioral events: {len(beh):,}  Participants: {beh['participant_id'].nunique()}")

## 2. Local-Calendar-Day Conversion (grouped by timezone for speed, not per-row apply)

In [ ]:
def to_local_date_series(df):
    out = pd.Series(index=df.index, dtype="object")
    for tz, grp in df.groupby("event_timezone"):
        try:
            out.loc[grp.index] = grp["event_timestamp_utc"].dt.tz_convert(tz).dt.date
        except Exception:
            out.loc[grp.index] = pd.NaT
    return out

created_timed = beh[(beh["event_type"] == "Task Created") & (beh["task_type"] == "timed")].copy()
created_timed["local_date"] = to_local_date_series(created_timed)
print(f"N_t^created source events: {len(created_timed):,}; conversion failures: {created_timed['local_date'].isna().sum()}")

def has_completed_tag(s):
    return pd.notna(s) and "task_completed" in s.split("|")

beh_upd = beh[beh["event_type"] == "Task Updated"].copy()
completion_events = beh_upd[beh_upd["update_type_list_str"].apply(has_completed_tag)].copy()
completion_events["local_date"] = to_local_date_series(completion_events)
print(f"N_t^completed source events: {len(completion_events):,}; conversion failures: {completion_events['local_date'].isna().sum()}")

created_timed = created_timed.dropna(subset=["local_date"])
completion_events = completion_events.dropna(subset=["local_date"])

## 3. Build Zero-Filled Daily Series per Participant

In [ ]:
def build_daily_series(df, pid):
    sub = df[df["participant_id"] == pid]
    if len(sub) == 0:
        return None
    daily = sub.groupby("local_date").size()
    daily.index = pd.to_datetime(daily.index)
    full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    daily = daily.reindex(full_range, fill_value=0)
    return daily

def series_stats(daily):
    n_active_days = int((daily > 0).sum())
    span_days = len(daily)
    is_eligible = (span_days >= MIN_SPAN_DAYS) and (n_active_days >= MIN_ACTIVE_DAYS)

    mean_N = float(daily.mean())
    var_N = float(daily.var(ddof=1)) if len(daily) > 1 else np.nan
    fano = var_N / mean_N if mean_N > 0 else np.nan

    diffs = daily.diff().dropna()
    mu_hat = float(diffs.mean()) if len(diffs) > 0 else np.nan
    sig_hat = float(diffs.std(ddof=1)) if len(diffs) > 1 else np.nan
    sigma_mu_ratio = (sig_hat / abs(mu_hat)) if (mu_hat is not None and abs(mu_hat) > 1e-9) else np.nan

    return dict(n_active_days=n_active_days, span_days=span_days, is_eligible=is_eligible,
                mean_N=round(mean_N, 4), Fano_factor=round(fano, 4) if not np.isnan(fano) else np.nan,
                mu_hat=round(mu_hat, 4) if not np.isnan(mu_hat) else np.nan,
                sig_hat=round(sig_hat, 4) if not np.isnan(sig_hat) else np.nan,
                sigma_mu_ratio=round(sigma_mu_ratio, 4) if sigma_mu_ratio is not None and not np.isnan(sigma_mu_ratio) else np.nan)

all_pids = sorted(beh["participant_id"].unique())
records = []
daily_long_records = []

for pid in all_pids:
    rec = {"participant_id": pid}

    d_created = build_daily_series(created_timed, pid)
    if d_created is not None:
        stats_c = series_stats(d_created)
        for k, v in stats_c.items():
            rec[f"created_{k}"] = v
        for date, val in d_created.items():
            daily_long_records.append({"participant_id": pid, "series": "created",
                                        "local_date": date.date(), "N_t": int(val)})
    else:
        for k in ["n_active_days","span_days","is_eligible","mean_N","Fano_factor","mu_hat","sig_hat","sigma_mu_ratio"]:
            rec[f"created_{k}"] = np.nan if k != "is_eligible" else False

    d_completed = build_daily_series(completion_events, pid)
    if d_completed is not None:
        stats_p = series_stats(d_completed)
        for k, v in stats_p.items():
            rec[f"completed_{k}"] = v
        for date, val in d_completed.items():
            daily_long_records.append({"participant_id": pid, "series": "completed",
                                        "local_date": date.date(), "N_t": int(val)})
    else:
        for k in ["n_active_days","span_days","is_eligible","mean_N","Fano_factor","mu_hat","sig_hat","sigma_mu_ratio"]:
            rec[f"completed_{k}"] = np.nan if k != "is_eligible" else False

    records.append(rec)

stop_df = pd.DataFrame(records)
daily_long = pd.DataFrame(daily_long_records)

print(f"Participants: {len(stop_df)}")
print(f"N_t^created eligible: {int(stop_df['created_is_eligible'].sum())} ({stop_df['created_is_eligible'].mean()*100:.1f}%)")
print(f"N_t^completed eligible: {int(stop_df['completed_is_eligible'].sum())} ({stop_df['completed_is_eligible'].mean()*100:.1f}%)")

## 4. Sanity Checks

In [ ]:
elig_c = stop_df[stop_df["created_is_eligible"]]
elig_p = stop_df[stop_df["completed_is_eligible"]]

print("N_t^created (eligible, N={}):" .format(len(elig_c)))
print(elig_c[["created_mean_N", "created_Fano_factor", "created_mu_hat", "created_sig_hat"]].describe().round(3))
print()
print("N_t^completed (eligible, N={}):".format(len(elig_p)))
print(elig_p[["completed_mean_N", "completed_Fano_factor", "completed_mu_hat", "completed_sig_hat"]].describe().round(3))
print()
print("Fano factor should be strictly positive (mean_N>0 required for eligibility-relevant rows):")
print("created:", (elig_c["created_Fano_factor"] > 0).all())
print("completed:", (elig_p["completed_Fano_factor"] > 0).all())

## 5. Descriptive Phenotype Labels (Part 5 of blueprint — illustrative only, NOT a clustering result)

In [ ]:
def label_phenotype(row, prefix):
    mu = row[f"{prefix}_mu_hat"]
    fano = row[f"{prefix}_Fano_factor"]
    smr = row[f"{prefix}_sigma_mu_ratio"]
    if pd.isna(mu) or pd.isna(fano):
        return np.nan
    return {"mu": mu, "fano": fano, "smr": smr}

for prefix, elig in [("created", elig_c), ("completed", elig_p)]:
    mu_col = f"{prefix}_mu_hat"
    fano_col = f"{prefix}_Fano_factor"
    smr_col = f"{prefix}_sigma_mu_ratio"

    mu_abs = elig[mu_col].abs()
    mu_terc = mu_abs.quantile([1/3, 2/3]).values
    fano_terc = elig[fano_col].quantile([1/3, 2/3]).values
    smr_terc = elig[smr_col].dropna().quantile([1/3, 2/3]).values if elig[smr_col].notna().sum() >= 3 else [np.nan, np.nan]

    def classify(r):
        mu, fano, smr = r[mu_col], r[fano_col], r[smr_col]
        if abs(mu) >= mu_terc[1] and mu > 0:
            return "growth-drift"
        if abs(mu) >= mu_terc[1] and mu < 0:
            return "decay-drift"
        if fano >= fano_terc[1]:
            return "bursty"
        if pd.notna(smr) and smr <= smr_terc[0]:
            return "stable"
        return "noise-dominated"

    stop_df.loc[elig.index, f"{prefix}_phenotype_label"] = elig.apply(classify, axis=1)

print("created series phenotype label counts:")
print(stop_df["created_phenotype_label"].value_counts(dropna=False))
print()
print("completed series phenotype label counts:")
print(stop_df["completed_phenotype_label"].value_counts(dropna=False))

## 6. Subscription Group Comparison (Part 6 of blueprint — observational, non-causal)

In [ ]:
sub_last = beh.sort_values("event_timestamp_utc").groupby("participant_id")["subscription_type_at_event"].last()
stop_df["subscription_type_last"] = stop_df["participant_id"].map(sub_last)

sub_summary = stop_df[stop_df["created_is_eligible"]].groupby("subscription_type_last")[
    ["created_mu_hat", "created_sig_hat", "created_Fano_factor"]
].agg(["mean", "median", "count"])
print(sub_summary.round(3))
print()
print("NOTE: purely observational. Free/pro is not randomized -- no causal claim.")

## 7. Save Outputs

In [ ]:
stop_df.to_csv(OUT / "participant_stop_parameters.csv", index=False)
daily_long.to_csv(OUT / "daily_series_long.csv", index=False)
sub_summary.to_csv(OUT / "subscription_group_summary.csv")
print("Saved:")
for f in ["participant_stop_parameters.csv", "daily_series_long.csv", "subscription_group_summary.csv"]:
    print(" ", f)

## 8. Figure 1 — Drift vs Diffusion Scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, prefix, title in [(axes[0], "created", "N_t^created"), (axes[1], "completed", "N_t^completed")]:
    d = stop_df[stop_df[f"{prefix}_is_eligible"]]
    colors = d[f"{prefix}_phenotype_label"].map({
        "growth-drift": "#55A868", "decay-drift": "#C44E52", "bursty": "#DD8452",
        "stable": "#4C72B0", "noise-dominated": "#8172B2"
    })
    ax.scatter(d[f"{prefix}_mu_hat"], d[f"{prefix}_sig_hat"], c=colors, s=50, alpha=0.8)
    ax.axvline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel("mu_hat (drift, tasks/day)")
    ax.set_ylabel("sig_hat (diffusion)")
    ax.set_title(title)
plt.tight_layout()
plt.savefig(FIG / "fig1_stop_mu_sigma_scatter.png", dpi=150)
plt.show()
print("Saved fig1_stop_mu_sigma_scatter.png")

## 9. Figure 2 — Fano Factor Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(elig_c["created_Fano_factor"].dropna(), bins=15, color="#4C72B0", edgecolor="white")
axes[0].axvline(1, color="red", ls="--", label="Poisson (Fano=1)")
axes[0].set_title("N_t^created Fano factor")
axes[0].legend()
axes[1].hist(elig_p["completed_Fano_factor"].dropna(), bins=15, color="#DD8452", edgecolor="white")
axes[1].axvline(1, color="red", ls="--", label="Poisson (Fano=1)")
axes[1].set_title("N_t^completed Fano factor")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_stop_fano_distribution.png", dpi=150)
plt.show()
print("Saved fig2_stop_fano_distribution.png")
print()
print(f"Share with Fano>1 (overdispersed/bursty) -- created: {(elig_c['created_Fano_factor']>1).mean()*100:.1f}%, "
      f"completed: {(elig_p['completed_Fano_factor']>1).mean()*100:.1f}%")

## 10. Figure 3 — Subscription Group Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["created_mu_hat", "created_sig_hat", "created_Fano_factor"]):
    data = [stop_df[(stop_df["created_is_eligible"]) & (stop_df["subscription_type_last"] == g)][metric].dropna()
            for g in ["free", "pro"]]
    ax.boxplot(data, labels=["free", "pro"])
    ax.set_title(metric)
plt.suptitle("Subscription group comparison (observational)")
plt.tight_layout()
plt.savefig(FIG / "fig3_stop_subscription_comparison.png", dpi=150)
plt.show()
print("Saved fig3_stop_subscription_comparison.png")

## 11. model_run_summary.md

In [ ]:
summary = f"""# Structured StoP -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/63_structured_stochastic_processes.ipynb
**Blueprint:** outputs/reports/structured/structured_stop_blueprint.md

## Eligibility
- N_t^created (timed-task creations, local day): {int(stop_df['created_is_eligible'].sum())}/{len(stop_df)} ({stop_df['created_is_eligible'].mean()*100:.1f}%)
- N_t^completed (completion-marking events, local day): {int(stop_df['completed_is_eligible'].sum())}/{len(stop_df)} ({stop_df['completed_is_eligible'].mean()*100:.1f}%)

## N_t^created (eligible, N={len(elig_c)})
- mu_hat: mean={elig_c['created_mu_hat'].mean():.3f}, median={elig_c['created_mu_hat'].median():.3f}
- sig_hat: mean={elig_c['created_sig_hat'].mean():.3f}
- Fano_factor: mean={elig_c['created_Fano_factor'].mean():.3f}, share>1 (bursty): {(elig_c['created_Fano_factor']>1).mean()*100:.1f}%

## N_t^completed (eligible, N={len(elig_p)})
- mu_hat: mean={elig_p['completed_mu_hat'].mean():.3f}, median={elig_p['completed_mu_hat'].median():.3f}
- sig_hat: mean={elig_p['completed_sig_hat'].mean():.3f}
- Fano_factor: mean={elig_p['completed_Fano_factor'].mean():.3f}, share>1 (bursty): {(elig_p['completed_Fano_factor']>1).mean()*100:.1f}%

## Descriptive phenotype labels (illustrative only, NOT a Step 2 clustering result)
created:
{stop_df['created_phenotype_label'].value_counts(dropna=False).to_string()}

completed:
{stop_df['completed_phenotype_label'].value_counts(dropna=False).to_string()}

## Subscription group comparison (observational only, no causal claim)
{sub_summary.round(3).to_string()}
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)